In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Script para recortar archivos CSV por rango de fechas.
Crea una copia de los datos originales con solo los años seleccionados.
"""

import os
import pandas as pd
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN (¡CÁMBIALO A TU GUSTO!)
# ============================================================================

# Ruta original donde están los CSVs (la misma que usas)
ORIGINAL_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_TFG_outliers/")

# Ruta donde se guardarán los archivos recortados
OUTPUT_DIR = os.path.expanduser("/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Rango de fechas deseado (incluye ambos extremos)
FECHA_INICIO = "2024-01-01"   # formato YYYY-MM-DD
FECHA_FIN    = "2025-12-31"   # puedes poner una fecha futura si quieres hasta el final

# Lista de archivos (la misma que proporcionaste)
station_paths = [
    "T1_E1_Alicante_outliers.csv",
    "T1_E2_Elda_outliers.csv",
    "T2_E1_Elche_outliers.csv",
    "T2_E2_Elda_outliers.csv",
]

# ============================================================================
# PROCESAMIENTO
# ============================================================================

def recortar_archivo(nombre_archivo):
    """Carga un CSV, filtra por fechas y guarda la versión recortada."""
    ruta_original = os.path.join(ORIGINAL_DIR, nombre_archivo)
    ruta_destino = os.path.join(OUTPUT_DIR, nombre_archivo)

    print(f"Procesando {nombre_archivo}...")

    try:
        # Cargar el CSV
        df = pd.read_csv(ruta_original)

        # Buscar la columna de fecha (puede llamarse 'datetime', 'fecha', etc.)
        # Ajusta el nombre si es diferente
        col_fecha = None
        for col in df.columns:
            if col.lower() in ['datetime', 'fecha', 'time', 'timestamp']:
                col_fecha = col
                break
        if col_fecha is None:
            print(f"  ⚠️ No se encontró columna de fecha en {nombre_archivo}. Se copia sin filtrar.")
            df.to_csv(ruta_destino, index=False)
            return

        # Convertir a datetime
        df[col_fecha] = pd.to_datetime(df[col_fecha], errors='coerce')
        df = df.dropna(subset=[col_fecha])   # eliminar filas con fecha inválida

        # Filtrar por rango
        mask = (df[col_fecha] >= pd.to_datetime(FECHA_INICIO)) & (df[col_fecha] <= pd.to_datetime(FECHA_FIN))
        df_filtrado = df.loc[mask].copy()

        # Guardar
        df_filtrado.to_csv(ruta_destino, index=False)
        print(f"  ✅ {len(df_filtrado)} filas guardadas (original: {len(df)})")

    except Exception as e:
        print(f"  ❌ Error: {e}")

# Recorrer todos los archivos
for archivo in station_paths:
    recortar_archivo(archivo)

print("\nProceso completado. Revisa la carpeta:")
print(OUTPUT_DIR)

Procesando T1_E1_Alicante_outliers.csv...


/var/folders/y_/8n8k4c3130sbkdjs9tb0_pv80000gp/T/ipykernel_2556/1481706282.py:48: DtypeWarning: Columns (0: physical_reason) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(ruta_original)


  ✅ 17521 filas guardadas (original: 140256)
Procesando T1_E2_Elda_outliers.csv...
  ✅ 17521 filas guardadas (original: 140256)
Procesando T2_E1_Elche_outliers.csv...
  ✅ 17521 filas guardadas (original: 140256)
Procesando T2_E2_Elda_outliers.csv...
  ✅ 17521 filas guardadas (original: 140256)

Proceso completado. Revisa la carpeta:
/Volumes/copia seguridad1/TFG_Prueba/Datos_iniciales/
